In [ ]:
# ============================================================
# COACH — Session Start  (do not remove this cell)
# ============================================================
import sys, os
sys.path.insert(0, os.path.expanduser('~/Desktop/applied-ai-research'))
from coach.notebook_widgets import render_session_start
_SESSION = render_session_start(
    module_id="prompt-engineering",
    notebook_name="03_prompt_templates.ipynb"
)

# Prompt Templates — Hands-On

In this notebook, you will build prompt templates using pure Python string formatting.
No API keys or LLM calls needed — this notebook runs entirely on your machine.

If you have not read [prompt-templates.md](./prompt-templates.md) yet, start there first —
it explains the concepts. This notebook shows them in code.

## Setup

In [ ]:
from string import Template
import textwrap

print("Setup complete. No external dependencies needed.")
print("This notebook uses pure Python string formatting.")

## Part 1: From Hardcoded Prompts to Templates

We start with a hardcoded prompt and transform it into a reusable template,
step by step.

In [ ]:
# ── The hardcoded way (not reusable) ────────────────────────────────
prompt_monday = "Summarize this tech article in 3 bullet points for beginners: [article about AI]"
prompt_tuesday = "Summarize this health article in 3 bullet points for beginners: [article about sleep]"
prompt_wednesday = "Summarize this sports article in 3 bullet points for beginners: [article about tennis]"

print("HARDCODED PROMPTS (not reusable):")
print(f"  Monday:    {prompt_monday}")
print(f"  Tuesday:   {prompt_tuesday}")
print(f"  Wednesday: {prompt_wednesday}")
print()
print("Problem: we are writing almost the same thing every time.")
print("Only 'tech/health/sports' and the article text change.")

In [ ]:
# ── The template way (reusable) ─────────────────────────────────────
template = "Summarize this {topic} article in {n} bullet points for {audience}: {text}"

# Now we fill in the blanks for each day
prompt_monday = template.format(
    topic="tech",
    n=3,
    audience="beginners",
    text="[article about AI]",
)
prompt_tuesday = template.format(
    topic="health",
    n=3,
    audience="beginners",
    text="[article about sleep]",
)
prompt_wednesday = template.format(
    topic="sports",
    n=3,
    audience="beginners",
    text="[article about tennis]",
)

print("TEMPLATE:")
print(f"  {template}")
print()
print("GENERATED PROMPTS:")
print(f"  Monday:    {prompt_monday}")
print(f"  Tuesday:   {prompt_tuesday}")
print(f"  Wednesday: {prompt_wednesday}")
print()
print("Same result, but we wrote the template once and reused it three times.")

## Part 2: The Four Parts of a Good Template

A well-designed template has four parts: **Role, Context, Task, Format.**
Let's build one piece by piece.

In [ ]:
# ── Building a template piece by piece ──────────────────────────────

# Part 1: Role
role_part = "You are a {role}."
print("Part 1 — ROLE:")
print(f"  {role_part}")
print()

# Part 2: Context
context_part = "Here is the {doc_type}:\n{content}"
print("Part 2 — CONTEXT:")
print(f"  {context_part}")
print()

# Part 3: Task
task_part = "Please {action} the above {doc_type}."
print("Part 3 — TASK:")
print(f"  {task_part}")
print()

# Part 4: Format
format_part = "Output as {output_format}."
print("Part 4 — FORMAT:")
print(f"  {format_part}")
print()

# Assemble all four parts
full_template = f"{role_part}\n\n{context_part}\n\n{task_part}\n\n{format_part}"
print("COMPLETE TEMPLATE:")
print("-" * 50)
print(full_template)
print("-" * 50)

In [ ]:
# ── Fill in the template with real values ───────────────────────────
prompt = full_template.format(
    role="senior financial analyst",
    doc_type="quarterly earnings report",
    content="Revenue was $5.2B, up 12% year-over-year. Net income grew 8%. "
            "The company launched two new products and expanded into 3 new markets.",
    action="analyze",
    output_format="3 bullet points with one-sentence summary at the end",
)

print("FILLED-IN PROMPT:")
print("=" * 50)
print(prompt)
print("=" * 50)
print()
print("This prompt is ready to send to an LLM.")

## Part 3: Variables with Defaults

Sometimes a variable has a sensible default that you only override when needed.
Python's `str.format` does not support defaults natively, but we can build a
simple helper.

In [ ]:
# ── Template with defaults ──────────────────────────────────────────
DEFAULTS = {
    "role": "helpful assistant",
    "num_points": 3,
    "audience": "general audience",
    "language": "English",
}


def fill_template(template: str, **kwargs) -> str:
    """Fill a template with provided values, falling back to defaults."""
    merged = {**DEFAULTS, **kwargs}
    return template.format(**merged)


template_with_defaults = (
    "You are a {role}.\n\n"
    "Summarize the following text in {num_points} bullet points "
    "for a {audience}.\n\n"
    "Respond in {language}.\n\n"
    "Text: {text}"
)

# Usage 1: minimal — only provide the required variable
prompt_minimal = fill_template(
    template_with_defaults,
    text="The new iPhone features a titanium body and improved camera system.",
)

# Usage 2: override some defaults
prompt_custom = fill_template(
    template_with_defaults,
    text="The new iPhone features a titanium body and improved camera system.",
    role="tech journalist",
    num_points=5,
    audience="tech-savvy readers",
)

print("MINIMAL (uses all defaults):")
print(prompt_minimal)
print()
print("=" * 50)
print()
print("CUSTOM (overrides some defaults):")
print(prompt_custom)

## Part 4: Building a Template Library

A template library is a collection of your best templates, organized by category.
Here we build a simple one as a Python dictionary.

In [ ]:
# ── Template library ────────────────────────────────────────────────
TEMPLATE_LIBRARY = {
    "summarize": {
        "description": "Summarize text into bullet points",
        "template": (
            "Summarize the following text in {num_points} bullet points "
            "for a {audience} audience:\n\n{text}"
        ),
        "required_vars": ["text"],
        "defaults": {"num_points": 3, "audience": "general"},
    },
    "code_review": {
        "description": "Review code for bugs and improvements",
        "template": (
            "You are a senior {language} developer.\n\n"
            "Review this code for bugs, performance, and readability:\n\n"
            "{code}\n\n"
            "Format your response as:\n"
            "- Bugs: [list]\n"
            "- Performance: [concerns]\n"
            "- Readability: [suggestions]\n"
            "- Improved code: [rewritten version]"
        ),
        "required_vars": ["language", "code"],
        "defaults": {},
    },
    "classify": {
        "description": "Classify text into categories",
        "template": (
            "Classify the following text into one of these categories: {categories}.\n\n"
            "Text: {text}\n\n"
            "Category:"
        ),
        "required_vars": ["text", "categories"],
        "defaults": {},
    },
    "translate": {
        "description": "Translate text between languages",
        "template": (
            "Translate the following {source} text to {target}.\n\n"
            "{text}"
        ),
        "required_vars": ["source", "target", "text"],
        "defaults": {},
    },
}

# ── Show the library ────────────────────────────────────────────────
print("TEMPLATE LIBRARY")
print("=" * 50)
for name, info in TEMPLATE_LIBRARY.items():
    req = ", ".join(info["required_vars"])
    defaults = ", ".join(f"{k}={v}" for k, v in info["defaults"].items()) or "none"
    print(f"  {name:15s} — {info['description']}")
    print(f"  {'':15s}   Required: {req}")
    print(f"  {'':15s}   Defaults: {defaults}")
    print()

In [ ]:
# ── Use the library ─────────────────────────────────────────────────
def use_template(name: str, **kwargs) -> str:
    """Look up a template by name and fill it in."""
    entry = TEMPLATE_LIBRARY[name]
    
    # Check required variables
    missing = [v for v in entry["required_vars"] if v not in kwargs]
    if missing:
        raise ValueError(f"Missing required variables: {missing}")
    
    # Merge defaults with provided values
    merged = {**entry["defaults"], **kwargs}
    return entry["template"].format(**merged)


# Example 1: Summarize
prompt1 = use_template(
    "summarize",
    text="Machine learning models are trained on large datasets...",
)
print("SUMMARIZE PROMPT:")
print(prompt1)
print()

# Example 2: Code review
prompt2 = use_template(
    "code_review",
    language="Python",
    code="def avg(nums): return sum(nums) / len(nums)",
)
print("CODE REVIEW PROMPT:")
print(prompt2)
print()

# Example 3: Classify
prompt3 = use_template(
    "classify",
    text="My credit card was charged twice",
    categories="BILLING, TECHNICAL, GENERAL",
)
print("CLASSIFY PROMPT:")
print(prompt3)

## Part 5: Combining Templates with Few-Shot and CoT

Templates become even more powerful when you combine them with the techniques
from the previous notebooks. Here we build a template that includes
few-shot examples and chain-of-thought instructions.

In [ ]:
# ── Few-Shot CoT template ───────────────────────────────────────────
fewshot_cot_template = """You are a {role}. Solve the following problem step by step.

Example:
Problem: {example_problem}
Solution:
{example_solution}

Now solve this:
Problem: {actual_problem}
Solution:"""

# Fill it in
prompt = fewshot_cot_template.format(
    role="math tutor",
    example_problem="A store has 20 apples. They sell half in the morning. How many are left?",
    example_solution=(
        "Step 1: Start with 20 apples.\n"
        "Step 2: Sell half: 20 / 2 = 10 sold.\n"
        "Step 3: Remaining: 20 - 10 = 10 apples.\n"
        "Answer: 10 apples."
    ),
    actual_problem="A bakery makes 36 cookies. They sell 2/3 in the morning and give away 4. How many are left?",
)

print("FEW-SHOT + COT + TEMPLATE:")
print("=" * 60)
print(prompt)
print("=" * 60)
print()
print("This prompt combines three techniques:")
print("  1. Template    — reusable structure with {variables}")
print("  2. Few-shot    — example problem + solution")
print("  3. CoT         — step-by-step reasoning in the example")

## Part 6: Validating Template Variables

A common mistake is forgetting to fill in a variable, or using the wrong variable name.
Let's build a simple validator.

In [ ]:
import re


def get_template_variables(template: str) -> list:
    """Extract all {variable} names from a template string."""
    return re.findall(r'\{(\w+)\}', template)


def validate_template(template: str, **kwargs) -> dict:
    """Check if all variables are provided and no extras are given."""
    required = set(get_template_variables(template))
    provided = set(kwargs.keys())
    
    missing = required - provided
    extra = provided - required
    
    return {
        "valid": len(missing) == 0,
        "missing": missing,
        "extra": extra,
        "required": required,
        "provided": provided,
    }


# ── Test the validator ───────────────────────────────────────────────
test_template = "You are a {role}. Summarize {text} in {num_points} points for {audience}."

print("Template:", test_template)
print(f"Variables found: {get_template_variables(test_template)}")
print()

# Test 1: all variables provided
result1 = validate_template(
    test_template,
    role="analyst", text="article", num_points=3, audience="beginners",
)
print(f"Test 1 (all provided): valid={result1['valid']}")

# Test 2: missing a variable
result2 = validate_template(
    test_template,
    role="analyst", text="article",
)
print(f"Test 2 (missing vars):  valid={result2['valid']}, missing={result2['missing']}")

# Test 3: extra variable
result3 = validate_template(
    test_template,
    role="analyst", text="article", num_points=3, audience="beginners", color="blue",
)
print(f"Test 3 (extra var):     valid={result3['valid']}, extra={result3['extra']}")

## Summary

What you just built:

1. **Hardcoded → Template**: turned repetitive prompts into a reusable template
2. **Four-part structure**: role + context + task + format = consistent results
3. **Defaults**: templates with sensible defaults that you override when needed
4. **Template library**: a dictionary of named templates you can look up and use
5. **Combined techniques**: templates + few-shot + CoT in a single prompt
6. **Validation**: a helper that catches missing or extra variables

For the concepts behind templates, read [prompt-templates.md](./prompt-templates.md).
Next notebook: [04_prompt_evaluation.ipynb](./04_prompt_evaluation.ipynb)

In [ ]:
# ============================================================
# COACH — Session End  (do not remove this cell)
# ============================================================
from coach.notebook_widgets import render_session_end
render_session_end(_SESSION)